# Large-Scale Dynamics of the Convection Zone and Tachocline — Implementation
# 대류층과 타코클라인의 대규모 역학 — 구현

**Paper**: Miesch, M.S. (2005), *Living Rev. Solar Phys.*, 2, 1.

1. **Solar differential rotation profile** — Helioseismic rotation from Figure 1 / 태양 차등 회전 프로파일
2. **Tachocline structure** — Error function fit and prolate shape / 타코클라인 구조
3. **Taylor-Proudman theorem & thermal wind** — Why Ω contours aren't cylindrical / 왜 Ω 등고선이 원통형이 아닌가
4. **Angular momentum transport** — Reynolds stress vs meridional circulation / 각운동량 수송
5. **Solar dynamo schematic** — α-Ω process visualization / 태양 다이나모 도식
6. **Tachocline confinement models** — Spiegel-Zahn vs Gough-McIntyre / 타코클라인 가둠 모델
7. **Convection scales & Reynolds numbers** — The modeling challenge / 대류 스케일과 모델링 도전
8. **Connection to modern simulations** — 현대 시뮬레이션과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from scipy.special import erf

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Solar constants
R_SUN = 6.96e10  # cm
OMEGA_0 = 2.6e-6  # rad/s, frame rotation rate (~414 nHz)

## Part 1: Solar Differential Rotation Profile (cf. Figure 1)
## 파트 1: 태양 차등 회전 프로파일 (Figure 1 비교)

Helioseismology reveals that the angular velocity $\Omega$ decreases monotonically from equator to pole by ~30%, with:
- Nearly **radial** contours at mid-latitudes in the convection zone
- A sharp transition at the **tachocline** ($r \sim 0.69 R_\odot$) to nearly uniform rotation
- A **near-surface shear layer** where $\Omega$ increases inward

일진학은 각속도 $\Omega$가 적도에서 극으로 ~30% 단조 감소하며, 중위도에서는 거의 **반경 방향** 등고선, **타코클라인**($r \sim 0.69 R_\odot$)에서의 급격한 전이, 그리고 표면 근처 전단층을 보여줍니다.

In [ ]:
# Model solar internal rotation profile based on helioseismic data
# 일진학 데이터 기반 태양 내부 회전 프로파일 모델

def solar_rotation(r_frac, lat_deg):
    """Model solar angular velocity Ω/2π in nHz.
    
    Based on helioseismic inversions (Thompson et al. 2003, Figure 1).
    
    Args:
        r_frac: Fractional radius r/R_sun (array).
        lat_deg: Latitude in degrees (scalar).
    
    Returns:
        Angular velocity Ω/2π in nHz.
    """
    lat = np.radians(lat_deg)
    
    # Surface differential rotation (Legendre polynomial fit)
    # Ω/2π = A + B*cos²θ + C*cos⁴θ  (θ = colatitude)
    # Typical values: A ≈ 460, B ≈ -62, C ≈ -67 nHz
    A, B, C = 460, -62, -67
    omega_surface = A + B * np.sin(lat)**2 + C * np.sin(lat)**4
    
    # Interior uniform rotation rate (radiative zone)
    omega_core = 430  # nHz (intermediate between equator and ~30° latitude)
    
    # Tachocline transition (error function, Eq. 1 of Miesch)
    r_t = 0.693  # tachocline center (equator)
    # Prolate: tachocline shifts outward at high latitudes
    r_t_lat = r_t + 0.024 * np.sin(lat)**2
    delta_t = 0.04  # tachocline width
    
    # Transition function
    f_tach = 0.5 * (1 + erf(2 * (r_frac - r_t_lat) / delta_t))
    
    # Near-surface shear layer (increases inward from surface to ~0.95 R)
    shear_amplitude = 15  # nHz
    f_shear = np.where(r_frac > 0.95,
                        shear_amplitude * (1 - r_frac) / 0.05,
                        0)
    
    omega = omega_core + (omega_surface - omega_core) * f_tach + f_shear
    return omega


# Create 2D rotation profile
r = np.linspace(0.5, 1.0, 300)
lat = np.linspace(-90, 90, 300)
R, LAT = np.meshgrid(r, lat)

# Compute rotation
OMEGA = np.zeros_like(R)
for i in range(len(lat)):
    OMEGA[i, :] = solar_rotation(r, lat[i])

# --- Plot: 2D rotation profile (Figure 1a style) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel a: 2D meridional cross-section
ax = axes[0]
# Convert to Cartesian for plotting
THETA = np.radians(90 - LAT)  # colatitude
X = R * np.sin(THETA)  # cylindrical radius
Y = R * np.cos(THETA)  # height above equator

# Only plot upper hemisphere
mask = LAT >= 0
levels = np.linspace(300, 475, 36)
cs = ax.contourf(X, Y, OMEGA, levels=levels, cmap="RdYlBu_r")
ax.contour(X, Y, OMEGA, levels=levels[::3], colors="k", linewidths=0.5, alpha=0.5)
plt.colorbar(cs, ax=ax, label="Ω/2π (nHz)")

# Draw boundaries
theta_arc = np.linspace(0, np.pi/2, 100)
for r_boundary, label, color in [(0.713, "CZ base", "black"),
                                   (0.693, "Tachocline", "red"),
                                   (1.0, "Surface", "gray")]:
    ax.plot(r_boundary * np.sin(theta_arc), r_boundary * np.cos(theta_arc),
            color=color, ls="--", lw=1.5, alpha=0.7)

ax.set_xlabel("r sin θ (R☉)")
ax.set_ylabel("r cos θ (R☉)")
ax.set_title("Solar Internal Rotation Profile\n태양 내부 회전 프로파일 (cf. Figure 1a)")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_aspect("equal")

# Panel b: Ω vs radius at selected latitudes (Figure 1b style)
ax = axes[1]
for lat_val in [0, 15, 30, 45, 60, 75]:
    omega_lat = solar_rotation(r, lat_val)
    ax.plot(r, omega_lat, lw=2, label=f"{lat_val}°")

ax.axvline(0.713, color="gray", ls="--", alpha=0.5, label="CZ base")
ax.axvline(0.693, color="red", ls="--", alpha=0.5, label="Tachocline")

ax.set_xlabel("r / R☉")
ax.set_ylabel("Ω/2π (nHz)")
ax.set_title("Ω vs Radius at Selected Latitudes\n위도별 Ω vs 반경 (cf. Figure 1b)")
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0.5, 1.0)
ax.set_ylim(300, 480)

plt.tight_layout()
plt.show()

print("Key features / 핵심 특징:")
print("1. Equator (0°) rotates ~30% faster than poles (75°)")
print("2. Nearly radial Ω contours at mid-latitudes in CZ")
print("3. Sharp tachocline transition at r ~ 0.69 R☉")
print("4. Near-surface shear layer (r > 0.95 R☉)")

## Part 2: Tachocline Structure — Error Function Fit
## 파트 2: 타코클라인 구조 — 오차 함수 맞춤

The tachocline transition is characterized by an error function (Eq. 1):

$$f(r; r_t, \Delta_t) = \frac{1}{2}\left\{1 + \mathrm{erf}\left[\frac{2(r - r_t)}{\Delta_t}\right]\right\}$$

Key parameters from helioseismic inversions:
- Center: $r_t \sim 0.693 \pm 0.003\,R_\odot$ at equator, shifting to $\sim 0.717 \pm 0.003\,R_\odot$ at 60° → **prolate** shape
- Width: $\Delta_t/R_\odot = 0.039 \pm 0.013$ at equator, possibly wider at high latitudes
- The convection zone base ($r_b = 0.713\,R_\odot$) lies **above** the tachocline center at the equator

In [ ]:
# Tachocline structure: error function fit and prolate geometry
# 타코클라인 구조: 오차 함수 맞춤과 편구 기하학

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel 1: Error function transition at different latitudes ---
ax = axes[0]
r = np.linspace(0.60, 0.80, 500)

for lat_deg, color in [(0, "red"), (30, "orange"), (60, "blue")]:
    lat = np.radians(lat_deg)
    r_t = 0.693 + 0.024 * np.sin(lat)**2
    delta_t = 0.039 + 0.003 * np.sin(lat)**2  # slightly wider at high lat
    f = 0.5 * (1 + erf(2 * (r - r_t) / delta_t))
    ax.plot(r, f, color=color, lw=2.5, label=f"Lat = {lat_deg}° (r_t = {r_t:.3f})")
    ax.axvline(r_t, color=color, ls=":", alpha=0.4)

ax.axvline(0.713, color="gray", ls="--", lw=2, alpha=0.6, label="CZ base (0.713)")
ax.set_xlabel("r / R☉")
ax.set_ylabel("Transition function f(r)")
ax.set_title("Tachocline Error Function (Eq. 1)\n타코클라인 오차 함수")
ax.legend(fontsize=9)
ax.set_xlim(0.62, 0.78)

# --- Panel 2: Tachocline shape in meridional cross-section ---
ax = axes[1]
lat_range = np.linspace(0, 90, 100)
r_t_profile = 0.693 + 0.024 * np.sin(np.radians(lat_range))**2
r_t_upper = r_t_profile + 0.02  # upper boundary (~ +Δ_t/2)
r_t_lower = r_t_profile - 0.02  # lower boundary

# Convert to Cartesian
theta = np.radians(90 - lat_range)
for r_arr, label, color, ls in [
    (r_t_profile, "Center", "red", "-"),
    (r_t_upper, "Upper", "red", "--"),
    (r_t_lower, "Lower", "red", "--"),
]:
    x = r_arr * np.sin(theta)
    y = r_arr * np.cos(theta)
    ax.plot(x, y, color=color, ls=ls, lw=2, label=label)

# CZ base (spherical)
x_cz = 0.713 * np.sin(theta)
y_cz = 0.713 * np.cos(theta)
ax.plot(x_cz, y_cz, "k--", lw=2, label="CZ base (spherical)")

# Surface
x_surf = 1.0 * np.sin(theta)
y_surf = 1.0 * np.cos(theta)
ax.plot(x_surf, y_surf, "gray", lw=1)

# Fill tachocline region
ax.fill_betweenx(r_t_upper * np.cos(theta), r_t_lower * np.sin(theta),
                  r_t_upper * np.sin(theta), alpha=0.2, color="red")

ax.set_xlabel("r sin θ (R☉)")
ax.set_ylabel("r cos θ (R☉)")
ax.set_title("Prolate Tachocline Shape\n편구 타코클라인 형태")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_xlim(0, 0.85)
ax.set_ylim(0, 0.85)

# --- Panel 3: Radial shear dv_φ/dr across tachocline ---
ax = axes[2]
r_fine = np.linspace(0.65, 0.75, 500)
dr = r_fine[1] - r_fine[0]

for lat_deg, color in [(0, "red"), (30, "orange"), (60, "blue")]:
    omega_profile = solar_rotation(r_fine, lat_deg)
    # v_phi = Ω * r * sin(θ) in m/s; dv_phi/dr ≈ dΩ/dr * r * sinθ
    domega_dr = np.gradient(omega_profile, dr)  # nHz per R_sun
    ax.plot(r_fine, domega_dr, color=color, lw=2, label=f"{lat_deg}°")

ax.axhline(0, color="gray", ls="-", alpha=0.3)
ax.axvline(0.693, color="red", ls=":", alpha=0.5)
ax.axvline(0.713, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("r / R☉")
ax.set_ylabel("dΩ/dr (nHz / R☉)")
ax.set_title("Radial Shear Across Tachocline\n타코클라인의 반경방향 전단")
ax.legend()
ax.set_xlim(0.65, 0.75)

plt.tight_layout()
plt.show()

print("\nKey: The tachocline is PROLATE — at equator it lies within/below the")
print("overshoot region, while at high latitudes it extends into the CZ.")
print("핵심: 타코클라인은 편구형 — 적도에서는 오버슈트 영역 내/아래에, 고위도에서는 CZ 안으로 연장됩니다.")

## Part 3: Taylor-Proudman Theorem vs Thermal Wind Balance
## 파트 3: Taylor-Proudman 정리 vs Thermal Wind 균형

The Taylor-Proudman theorem states that in a rapidly rotating, adiabatic fluid, angular velocity contours should be **cylindrical** ($\Omega = \Omega(\lambda)$ where $\lambda = r\sin\theta$). But helioseismology shows **nearly radial** contours. The resolution is the **thermal wind balance** (Eq. 11):

$$\mathbf{\Omega}_0 \cdot \nabla\Omega = \frac{g}{2C_P\lambda r}\frac{\partial\langle S\rangle}{\partial\theta}$$

Latitudinal entropy gradients (warm poles, cool equator by ~5 K) break cylindrical symmetry. This is established by anisotropic convective heat transport under rotation.

In [ ]:
# Taylor-Proudman vs Thermal Wind: cylindrical vs radial Ω contours
# Taylor-Proudman vs Thermal Wind: 원통형 vs 반경방향 Ω 등고선

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

theta_grid = np.linspace(0.01, np.pi/2, 200)
r_grid = np.linspace(0.71, 1.0, 200)
R_GRID, TH_GRID = np.meshgrid(r_grid, theta_grid)
LAMBDA = R_GRID * np.sin(TH_GRID)  # cylindrical radius

# --- Panel 1: Cylindrical rotation (Taylor-Proudman) ---
ax = axes[0]
# Ω depends only on λ = r sinθ
omega_eq = 460  # nHz at equator surface
omega_pole = 330  # nHz at pole
OMEGA_CYL = omega_pole + (omega_eq - omega_pole) * (LAMBDA / np.max(LAMBDA))**2

X = R_GRID * np.sin(TH_GRID)
Y = R_GRID * np.cos(TH_GRID)
levels = np.linspace(330, 470, 30)
cs = ax.contourf(X, Y, OMEGA_CYL, levels=levels, cmap="RdYlBu_r")
ax.contour(X, Y, OMEGA_CYL, levels=levels[::3], colors="k", linewidths=0.8)
# Draw cylindrical lines
for lam in [0.3, 0.5, 0.7, 0.9]:
    ax.axvline(lam, color="green", ls="--", alpha=0.5, lw=1)
ax.set_title("Taylor-Proudman: Cylindrical Ω\nΩ = Ω(λ) — 원통형")
ax.set_xlabel("r sin θ")
ax.set_ylabel("r cos θ")
ax.set_aspect("equal")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)

# --- Panel 2: Observed (nearly radial) rotation ---
ax = axes[1]
OMEGA_OBS = np.zeros_like(R_GRID)
for i in range(len(theta_grid)):
    lat_deg = 90 - np.degrees(theta_grid[i])
    OMEGA_OBS[i, :] = solar_rotation(r_grid, lat_deg)

cs = ax.contourf(X, Y, OMEGA_OBS, levels=levels, cmap="RdYlBu_r")
ax.contour(X, Y, OMEGA_OBS, levels=levels[::3], colors="k", linewidths=0.8)
# Draw radial lines
for angle in [15, 30, 45, 60, 75]:
    r_line = np.linspace(0.71, 1.0, 50)
    ax.plot(r_line * np.sin(np.radians(angle)), r_line * np.cos(np.radians(angle)),
            "green", ls="--", alpha=0.5, lw=1)
ax.set_title("Observed: Nearly Radial Ω\nΩ ≈ Ω(r, θ) — 반경방향")
ax.set_xlabel("r sin θ")
ax.set_aspect("equal")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)

# --- Panel 3: Required entropy gradient for thermal wind ---
ax = axes[2]
# From Eq. (11): Ω₀·∇Ω = g/(2C_p λr) ∂S/∂θ
# The required ∂S/∂θ to produce radial (not cylindrical) Ω

lat_arr = np.linspace(5, 85, 100)  # avoid poles
theta_arr = np.radians(90 - lat_arr)

# Approximate: departure from cylindrical → thermal wind → ∂T/∂θ ~ 5 K
# Temperature perturbation consistent with thermal wind balance (Figure 7)
delta_T = 5 * np.cos(2 * theta_arr)  # ~5 K amplitude, warm poles

# Also plot the normalized entropy gradient
dS_dtheta = -np.sin(2 * theta_arr) * 8e-5  # C_p^{-1} ∂S/∂θ (normalized)

ax.plot(lat_arr, delta_T, "r-", lw=2.5, label="ΔT (K)")
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Temperature perturbation (K)", color="red")
ax.tick_params(axis="y", labelcolor="red")
ax.set_title("Thermal Wind: Required ΔT\nthermal wind에 필요한 ΔT (cf. Figure 7)")
ax.axhline(0, color="gray", ls="-", alpha=0.3)
ax.legend(loc="upper right")

# Secondary axis for entropy gradient
ax2 = ax.twinx()
ax2.plot(lat_arr, dS_dtheta * 1e5, "b--", lw=2, label=r"$C_P^{-1}\partial S/\partial\theta$ ($\times 10^5$)")
ax2.set_ylabel(r"$C_P^{-1}\partial S/\partial\theta$ ($\times 10^{-5}$)", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()

print("Key insight / 핵심 통찰:")
print("Cylindrical Ω (Taylor-Proudman) requires ∂S/∂θ = 0 (perfectly adiabatic)")
print("Radial Ω (observed) requires warm poles (~5 K warmer than equator)")
print("This small ΔT is below current helioseismic sensitivity limits!")
print("원통형 Ω(Taylor-Proudman)는 완전 단열(∂S/∂θ=0)을 요구")
print("반경형 Ω(관측)는 따뜻한 극(적도보다 ~5K 높음)을 요구")
print("이 작은 ΔT는 현재 일진학 감도 한계 이하입니다!")

## Part 4: Tachocline Confinement Models — The Central Question
## 파트 4: 타코클라인 가둠 모델 — 핵심 질문

**Why is the tachocline so thin?** Without confinement, radiative spreading (Spiegel & Zahn 1992) would spread differential rotation deep into the radiative interior over the Sun's lifetime.

Two competing models:

1. **Spiegel & Zahn (1992)**: Anisotropic turbulence ($\nu_H \gg \nu_V$) confines via horizontal mixing:
$$\frac{\Delta_t}{r_t} \sim \left(\frac{\Omega}{N}\right)^{1/2}\left(\frac{\kappa_r}{\nu_H}\right)^{1/4} \tag{30}$$

2. **Gough & McIntyre (1998)**: Fossil poloidal magnetic field + gyroscopic pumping:
$$\frac{\Delta_t}{r_t} \sim \left(\frac{4\pi\rho\,\nu\eta}{r_t^2\,B_0^2}\right)^{1/4} \sim 5 \times 10^{-5}\,B_0^{-1/2} \tag{31}$$

In [ ]:
# Tachocline confinement: comparing the two models
# 타코클라인 가둠: 두 모델 비교

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Solar parameters at tachocline
r_t = 0.7  # R_sun
Omega = 2.7e-6  # rad/s
N = 1e-3  # Brunt-Väisälä frequency (s^-1) in radiative zone
kappa_r = 1e7  # cm^2/s, radiative diffusivity at tachocline
rho = 0.2  # g/cm^3
nu_mol = 5  # cm^2/s, molecular viscosity
eta_mol = 1e3  # cm^2/s, magnetic diffusivity

# --- Panel 1: Spiegel-Zahn model (Eq. 30) ---
ax = axes[0]
nu_H_range = np.logspace(3, 10, 100)  # horizontal turbulent viscosity
delta_SZ = (Omega / N)**0.5 * (kappa_r / nu_H_range)**0.25

ax.loglog(nu_H_range, delta_SZ, "b-", lw=2.5, label="Spiegel-Zahn (Eq. 30)")
ax.axhline(0.05, color="red", ls="--", lw=2, label="Observed: Δ_t/r_t ≈ 0.05")
ax.axhline(0.04, color="red", ls=":", alpha=0.5)
ax.axhline(0.06, color="red", ls=":", alpha=0.5)

# Mark the required ν_H
nu_H_required = kappa_r * (Omega / N)**2 / 0.05**4
ax.axvline(nu_H_required, color="green", ls="--", alpha=0.7)
ax.annotate(f"ν_H ≈ {nu_H_required:.1e} cm²/s\nrequired", 
            (nu_H_required, 0.1), fontsize=9, color="green",
            textcoords="offset points", xytext=(10, 0))

ax.set_xlabel("ν_H (cm² s⁻¹) — horizontal turbulent viscosity")
ax.set_ylabel("Δ_t / r_t")
ax.set_title("Spiegel-Zahn Model\n비등방 난류 가둠")
ax.legend(fontsize=10)
ax.set_ylim(1e-3, 1)
ax.set_xlim(1e3, 1e10)

# --- Panel 2: Gough-McIntyre model (Eq. 31) ---
ax = axes[1]
B0_range = np.logspace(-7, 1, 100)  # Gauss
r_t_cm = r_t * R_SUN
delta_GM = (4 * np.pi * rho * nu_mol * eta_mol / (r_t_cm**2 * B0_range**2))**0.25

# Convert to R_sun units
delta_GM_frac = delta_GM * R_SUN / r_t_cm  # actually need to be more careful
# Simplified: Δ_t/r_t ~ 5e-5 * B0^{-1/2} from the paper
delta_GM_simple = 5e-5 * B0_range**(-0.5)

ax.loglog(B0_range, delta_GM_simple, "r-", lw=2.5, label="Gough-McIntyre (Eq. 31)")
ax.axhline(0.05, color="blue", ls="--", lw=2, label="Observed: Δ_t/r_t ≈ 0.05")
ax.axhline(0.04, color="blue", ls=":", alpha=0.5)

# Mark key field strengths
for B_val, label in [(1e-6, "10⁻⁶ G"), (1e-4, "10⁻⁴ G"), (1, "1 G")]:
    delta_at_B = 5e-5 * B_val**(-0.5)
    ax.plot(B_val, delta_at_B, "ko", markersize=8)
    ax.annotate(f"B₀={label}\nΔ/r={delta_at_B:.3f}", (B_val, delta_at_B),
                fontsize=8, textcoords="offset points", xytext=(10, 5))

ax.set_xlabel("B₀ (G) — fossil poloidal field strength")
ax.set_ylabel("Δ_t / r_t")
ax.set_title("Gough-McIntyre Model\n화석 자기장 가둠")
ax.legend(fontsize=10)
ax.set_ylim(1e-4, 10)
ax.set_xlim(1e-7, 10)

plt.tight_layout()
plt.show()

# Comparison table
print("=" * 60)
print("Tachocline Confinement Model Comparison")
print("타코클라인 가둠 모델 비교")
print("=" * 60)
print(f"{'':20s} {'Spiegel-Zahn':>18s} {'Gough-McIntyre':>18s}")
print(f"{'Mechanism':20s} {'Aniso. turbulence':>18s} {'Fossil B field':>18s}")
print(f"{'Key parameter':20s} {'ν_H ~ 3×10⁶ cm²/s':>18s} {'B₀ ~ 10⁻⁶–1 G':>18s}")
print(f"{'Timescale':20s} {'~10⁵ yr':>18s} {'~10⁶ yr':>18s}")
print(f"{'Uniform interior?':20s} {'Partial':>18s} {'Yes (Ferraro)':>18s}")
print(f"{'Weakness':20s} {'Cannot enforce':>18s} {'Fossil field':>18s}")
print(f"{'':20s} {'uniform rotation':>18s} {'stability unclear':>18s}")

## Part 5: Summary — The Modeling Challenge & Modern Status
## 파트 5: 요약 — 모델링 도전과 현대 상태

| Challenge (2005) | Status (2020s) | 현대 상태 |
|---|---|---|
| Polar vortex problem | Partially resolved with higher resolution (Hotta et al. 2022) | 고해상도로 부분 해결 |
| ~30% angular velocity contrast | Achieved in many simulations | 많은 시뮬레이션에서 달성 |
| Nearly radial Ω contours | Achieved via thermal wind (Brun et al. 2011) | thermal wind으로 달성 |
| Self-consistent tachocline | Still not achieved in global sims | 여전히 전구 시뮬레이션에서 미달성 |
| Torsional oscillations | Reproduced in some models with magnetic feedback | 일부 모델에서 자기 피드백으로 재현 |
| Cyclic dynamo in 3D | Achieved by several groups (Käpylä et al., Hotta et al.) | 여러 그룹이 달성 |
| ASH code ~1000³ grid | Modern codes: $10^{10}$+ grid points (Hotta 2023) | 현대 코드: 10¹⁰+ 격자점 |

The field has advanced enormously since 2005, but the **tachocline confinement** and **complete solar cycle reproduction** in a single self-consistent simulation remain frontier challenges.

이 분야는 2005년 이후 엄청나게 발전했지만, **타코클라인 가둠**과 **완전한 태양 주기 재현**을 하나의 자기일관적 시뮬레이션에서 달성하는 것은 여전히 최전선의 도전과제입니다.